<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/synth_suite_bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Synth Ecosystem Bridge: synth-pdb & synth-saxs

This tutorial demonstrates how to leverage the interoperability of the Synth suite. We will:
1. Generate a synthetic protein structure using `synth-pdb`.
2. Perform a brief physics-based structural relaxation.
3. Compute and visualize the resulting SAXS profile using `synth-saxs`.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/elkins-lab/synth-saxs.git synth-pdb biotite matplotlib
else:
    sys.path.append("../../")

In [ ]:
from synth_pdb import EnergyMinimizer, PeptideGenerator

from synth_saxs import calculate_saxs_profile, plot_saxs_results

## 1. Generating a Synthetic Protein Structure
We use `synth-pdb` to rapidly assemble an ideal $\alpha$-helical protein from a sequence.

In [ ]:
# Define a 40-residue sequence
sequence = "ACDEFGHIKLMNPQRSTVWY" * 2

print("Generating structure...")
generator = PeptideGenerator(sequence)
result = generator.generate(secondary_structure="helix")
structure = result.structure

print(f"Generated structure with {len(structure)} atoms.")

## 2. Relaxing the Structure
Idealized structures often have unnatural bond geometries. We use `synth-pdb`'s `EnergyMinimizer` to relax the structure in an implicit solvent model.

In [ ]:
print("Relaxing structure...")
minimizer = EnergyMinimizer()
relaxed_structure, info = minimizer.minimize(structure)

print("Relaxation complete.")

## 3. Simulating the SAXS Profile
We will compute the SAXS profiles for both the idealized and relaxed structures to see if the subtle compaction during relaxation affects the scattering curve.

In [ ]:
# Unrelaxed structure
q_ideal, i_ideal = calculate_saxs_profile(structure, q_min=0.01, q_max=0.3, n_points=100)

# Relaxed structure
q_relax, i_relax = calculate_saxs_profile(relaxed_structure, q_min=0.01, q_max=0.3, n_points=100)

## 4. Visualizing the Results
Finally, we use `synth-saxs` to plot the standard SAXS curves.

In [ ]:
plot_saxs_results(q_relax, i_relax, plot_type="all")